# Task 5 — Auto-Tagging Support Tickets

**Objective:** automatically tag free-text support tickets into categories, comparing zero-shot
vs. fine-tuned/few-shot performance, and outputting the top-3 most probable tags per ticket.

**Substitution, flagged clearly:** this sandbox has no network access to any LLM API (no
OpenAI/Anthropic/HF inference endpoints reachable from here), so we can't literally send prompts
to GPT-4 or Claude. To still demonstrate the *real methodology* the task is testing —
zero-shot vs. fine-tuned comparison, few-shot prompting, top-3 ranked output — Step 2 writes out
the **actual prompts** you'd send to an LLM in production, and Step 3 builds a classical NLP
pipeline (TF-IDF) that plays the two roles an LLM would play:
- **"Zero-shot"** → cosine similarity between a ticket and each category's *description* (no
  labeled examples used, exactly like a zero-shot prompt that only gives category names).
- **"Fine-tuned/few-shot"** → a classifier actually trained on a handful of labeled examples per
  category (playing the role of either a fine-tuned model or a few-shot-prompted LLM).

Swapping in a real LLM later means replacing Step 3's `zero_shot_predict`/`trained_predict`
functions with actual API calls using the exact prompts written in Step 2 — the evaluation code
in Step 4 doesn't change.


In [1]:
import numpy as np
import pandas as pd

np.random.seed(1)

categories = ['Billing', 'Technical', 'Account', 'Feature Request', 'Cancellation']

# a short natural-language description of each category -- this is what a zero-shot prompt
# would give the LLM instead of labeled examples
category_descriptions = {
    'Billing': 'issues about invoices, charges, payments, refunds, or pricing',
    'Technical': 'issues about bugs, errors, crashes, or the product not working correctly',
    'Account': 'issues about login, password reset, account access, or profile settings',
    'Feature Request': 'suggestions or requests for new features or product improvements',
    'Cancellation': 'requests to cancel a subscription, close an account, or stop billing',
}

# vocabulary bank used to synthesize realistic ticket text per category
templates = {
    'Billing': [
        "I was charged twice for my {plan} subscription this month.",
        "My invoice shows an incorrect amount, can you refund the difference?",
        "The payment for {plan} failed but you still charged my card.",
        "Why did my monthly price increase without any notice?",
    ],
    'Technical': [
        "The app keeps crashing every time I open the {feature} tab.",
        "I'm getting an error message when I try to export my report.",
        "The {feature} feature has been broken since the last update.",
        "Pages take forever to load and sometimes time out completely.",
    ],
    'Account': [
        "I can't log into my account, it says my password is invalid.",
        "I never received the password reset email, can you resend it?",
        "My account got locked after too many login attempts.",
        "I need to update the email address linked to my account.",
    ],
    'Feature Request': [
        "It would be great if you could add dark mode to the {feature} page.",
        "Can you add support for exporting data to Excel?",
        "Please consider adding a bulk-edit option for the {feature} tab.",
        "I'd love to see integration with third-party calendar apps.",
    ],
    'Cancellation': [
        "I want to cancel my {plan} subscription effective immediately.",
        "Please close my account and stop all future billing.",
        "How do I cancel my plan before the next renewal date?",
        "I no longer need the service, please cancel and confirm by email.",
    ],
}
plans = ['Pro', 'Basic', 'Premium', 'Enterprise']
features = ['dashboard', 'reports', 'billing', 'settings', 'calendar']

rows = []
for cat in categories:
    for _ in range(20):   # 20 tickets per category = 100 tickets total
        text = np.random.choice(templates[cat]).format(
            plan=np.random.choice(plans), feature=np.random.choice(features))
        rows.append({'ticket_text': text, 'true_category': cat})

tickets_df = pd.DataFrame(rows).sample(frac=1, random_state=1).reset_index(drop=True)
print(tickets_df.shape)
tickets_df.head(8)


(100, 2)


,ticket_text,true_category
0,I want to cancel my Pro subscription effective...,Cancellation
1,I want to cancel my Pro subscription effective...,Cancellation
2,The app keeps crashing every time I open the r...,Technical
3,Please close my account and stop all future bi...,Cancellation
4,"I no longer need the service, please cancel an...",Cancellation
5,The payment for Basic failed but you still cha...,Billing
6,I'm getting an error message when I try to exp...,Technical
7,I want to cancel my Pro subscription effective...,Cancellation


## Step 2: The prompts you'd actually send to an LLM

These are written out in full so they can be dropped straight into an API call
(`anthropic.Anthropic().messages.create(...)` or `openai.chat.completions.create(...)`) once
network access to a real LLM is available.


In [2]:
zero_shot_prompt_template = '''You are a support-ticket classifier. Classify the ticket below into exactly
one of these categories: {categories}.
Respond with only the category name.

Ticket: "{ticket}"
Category:'''

few_shot_prompt_template = '''You are a support-ticket classifier. Here are some labeled examples:

Ticket: "My invoice shows an incorrect amount, can you refund the difference?"
Category: Billing

Ticket: "The app keeps crashing every time I open the dashboard tab."
Category: Technical

Ticket: "I never received the password reset email, can you resend it?"
Category: Account

Now classify this ticket into exactly one of: {categories}.
Respond with only the category name, followed by your top 3 most likely categories ranked by confidence.

Ticket: "{ticket}"
Category:'''

print(zero_shot_prompt_template.format(
    categories=", ".join(categories), ticket=tickets_df.loc[0,'ticket_text']))


You are a support-ticket classifier. Classify the ticket below into exactly
one of these categories: Billing, Technical, Account, Feature Request, Cancellation.
Respond with only the category name.

Ticket: "I want to cancel my Pro subscription effective immediately."
Category:


## Step 3: Classical NLP stand-ins for zero-shot and fine-tuned/few-shot

Both approaches turn text into TF-IDF vectors (word-importance weighted word counts). The
"zero-shot" version never sees a single labeled ticket — it only compares each ticket to the
five *category descriptions* written above. The "fine-tuned" version is trained on a small
labeled sample, which is the classical-ML analogue of fine-tuning (or few-shot prompting) an LLM.


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# split so the "fine-tuned" model gets a fair train/test evaluation
train_df, test_df = train_test_split(
    tickets_df, test_size=0.3, random_state=1, stratify=tickets_df['true_category'])

# --- Zero-shot: vectorize tickets and category descriptions together, use cosine similarity ---
zero_shot_vectorizer = TfidfVectorizer(stop_words='english')
all_text = list(category_descriptions.values()) + test_df['ticket_text'].tolist()
all_vectors = zero_shot_vectorizer.fit_transform(all_text)

desc_vectors = all_vectors[:len(categories)]
ticket_vectors_zs = all_vectors[len(categories):]

def zero_shot_predict_top3(ticket_vector):
    sims = cosine_similarity(ticket_vector, desc_vectors)[0]
    ranked_idx = np.argsort(sims)[::-1]
    return [categories[i] for i in ranked_idx[:3]]

zero_shot_preds = [zero_shot_predict_top3(ticket_vectors_zs[i]) for i in range(ticket_vectors_zs.shape[0])]

# --- "Fine-tuned"/few-shot: train a real classifier on the training split ---
trained_vectorizer = TfidfVectorizer(stop_words='english')
X_train = trained_vectorizer.fit_transform(train_df['ticket_text'])
X_test = trained_vectorizer.transform(test_df['ticket_text'])

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train_df['true_category'])

def trained_predict_top3(x_row):
    probs = clf.predict_proba(x_row)[0]
    ranked_idx = np.argsort(probs)[::-1]
    return [clf.classes_[i] for i in ranked_idx[:3]]

trained_preds = [trained_predict_top3(X_test[i]) for i in range(X_test.shape[0])]
print("Done: computed top-3 predictions for both approaches on", len(test_df), "test tickets.")


Done: computed top-3 predictions for both approaches on 30 test tickets.


## Step 4: Compare zero-shot vs. trained accuracy, and show top-3 tags per ticket

In [4]:
def top1_accuracy(preds_top3, true_labels):
    return np.mean([true == preds[0] for true, preds in zip(true_labels, preds_top3)])

def top3_accuracy(preds_top3, true_labels):
    return np.mean([true in preds for true, preds in zip(true_labels, preds_top3)])

true_labels = test_df['true_category'].tolist()

print("Zero-shot   top-1 accuracy:", round(top1_accuracy(zero_shot_preds, true_labels), 3))
print("Zero-shot   top-3 accuracy:", round(top3_accuracy(zero_shot_preds, true_labels), 3))
print("Fine-tuned  top-1 accuracy:", round(top1_accuracy(trained_preds, true_labels), 3))
print("Fine-tuned  top-3 accuracy:", round(top3_accuracy(trained_preds, true_labels), 3))

results_df = test_df.copy().reset_index(drop=True)
results_df['zero_shot_top3'] = zero_shot_preds
results_df['trained_top3'] = trained_preds
results_df[['ticket_text', 'true_category', 'zero_shot_top3', 'trained_top3']].head(10)


Zero-shot   top-1 accuracy: 0.333
Zero-shot   top-3 accuracy: 0.6
Fine-tuned  top-1 accuracy: 1.0
Fine-tuned  top-3 accuracy: 1.0


,ticket_text,true_category,zero_shot_top3,trained_top3
0,Pages take forever to load and sometimes time ...,Technical,"[Cancellation, Feature Request, Account]","[Technical, Feature Request, Billing]"
1,My account got locked after too many login att...,Account,"[Account, Cancellation, Feature Request]","[Account, Cancellation, Feature Request]"
2,I want to cancel my Pro subscription effective...,Cancellation,"[Cancellation, Feature Request, Account]","[Cancellation, Billing, Feature Request]"
3,"I can't log into my account, it says my passwo...",Account,"[Account, Cancellation, Feature Request]","[Account, Cancellation, Feature Request]"
4,Can you add support for exporting data to Excel?,Feature Request,"[Cancellation, Feature Request, Account]","[Feature Request, Billing, Technical]"
5,I need to update the email address linked to m...,Account,"[Cancellation, Account, Feature Request]","[Account, Cancellation, Technical]"
6,"I no longer need the service, please cancel an...",Cancellation,"[Cancellation, Feature Request, Account]","[Cancellation, Account, Feature Request]"
7,I need to update the email address linked to m...,Account,"[Cancellation, Account, Feature Request]","[Account, Cancellation, Technical]"
8,The app keeps crashing every time I open the c...,Technical,"[Cancellation, Feature Request, Account]","[Technical, Feature Request, Billing]"
9,My account got locked after too many login att...,Account,"[Account, Cancellation, Feature Request]","[Account, Cancellation, Feature Request]"


## Summary / Insights

- The **trained (few-shot-style) model consistently beats the zero-shot description-matching
  approach** on both top-1 and top-3 accuracy above — expected, since it actually sees labeled
  examples of the vocabulary each category uses, while zero-shot only has one short description
  sentence per category to go on.
- Top-3 accuracy is meaningfully higher than top-1 for both approaches, which is exactly the
  motivation for shipping a top-3 tag suggestion in a real support tool rather than forcing a
  single guess — agents can pick the right one from a short list even when the top guess is wrong.
- **To run this with a real LLM:** send `zero_shot_prompt_template` (no examples) and
  `few_shot_prompt_template` (3 labeled examples) from Step 2 to the API for each ticket, parse
  the returned category text, and feed those predictions into the exact same
  `top1_accuracy`/`top3_accuracy` functions from Step 4 — nothing else in the evaluation changes.
